# Pipeline Architecture: Lambda & Kappa
**Day 3 — Technology Module**

---

<div style="padding:15px;border-left:8px solid #f7971e;background:#fff8e1;border-radius:4px;">
<strong>Core Insight:</strong> Data pipeline architecture boils down to: batch vs streaming vs hybrid.
Lambda architecture separates batch (complete, slow) from streaming (fast, approximate).
Kappa simplifies this to one codebase by treating everything as a stream.
Understanding the trade-offs between them is a senior-level interview topic.
</div>

## Lambda Architecture

```
Source Data
    ↓
    ├── BATCH LAYER (Spark/Hadoop)
    │   Process ALL historical data
    │   Hours/days latency, complete accuracy
    │   Output: batch views
    │
    └── SPEED LAYER (Kafka + Flink/Kinesis)
        Process RECENT data in real-time
        Seconds latency, may be approximate
        Output: real-time views
              ↓
        SERVING LAYER (merges batch + speed views)
              ↓
        Query (reads from both; speed fills recency gap)
```

**Lambda pros:** Batch layer is always correct; speed layer gives low-latency recency.
**Lambda cons:** Two codebases (batch + streaming) for the same logic — they drift apart over time.

## Kappa Architecture (Simplified Lambda)

```
Source Data → Kafka (append-only log, full history retained)
                ↓
        Stream Processing (Flink / Spark Streaming)
        ONE codebase — handles both real-time and reprocessing
                ↓
        Serving Layer (Cassandra, DynamoDB, S3)
                ↓
        Query

Reprocessing historical data:
  → Replay from Kafka beginning with updated code
  → Same pipeline processes history as if it were new events
```

**Kappa pros:** One codebase, simpler ops, reprocessing = just replay from Kafka.
**Kappa cons:** Keeping full history in Kafka is expensive; not all stream processors handle complex batch semantics well.

**When to choose Kappa over Lambda:**
- Modern stream processor (Flink, Spark Structured Streaming) handles your batch complexity
- You want to avoid maintaining two separate pipelines
- Reprocessing is common (schema changes, bug fixes)

## Late Data — The Hard Problem

**The scenario:** Events that arrive after their expected processing window has closed.
Example: mobile device offline for 30 minutes, then sends all events when reconnected.

**Solutions:**
1. **Watermarks:** Tell the stream processor "tolerate up to N minutes of late data." Events within the watermark are processed in-window; beyond it, trigger the window result.
2. **Upsert to target:** Accept late events and update the serving layer record rather than ignoring them.
3. **Reprocess from source:** For large late-data batches, replay the relevant time window.

```
Event time: 14:00     (when the event actually occurred)
Processing time: 14:35 (when it arrived at the processor)
Lateness: 35 minutes

Watermark = 30 min → this event is LATE, window already closed
Watermark = 60 min → this event is within tolerance, processed normally
```

## Real Example — Telemetry Pipeline at Scale

```
6,000 APM agents (CA APM, AppDynamics, Dynatrace, BMC TrueSight)
    │
    ▼
Kafka (10 partitions, 7-day retention, keyed by server_id)
    │
    ├──→ Flink job (real-time)
    │       5-minute tumbling windows
    │       Compute avg/max CPU per server
    │       Threshold alerts → PagerDuty
    │       Write real-time views → Redis (30-second TTL)
    │
    └──→ S3 sink (micro-batch, 5-min)
            Parquet files, partitioned by date + env
                │
                ▼
            Glue Crawler (auto-update catalog)
                │
                ├──→ Athena (ad-hoc SQL, capacity team)
                └──→ Daily Glue ETL
                        → Aggregate summaries
                        → Prophet forecasting input
                        → QuickSight dashboard
```

**This is effectively Kappa:** One source (Kafka), two consumers (real-time + S3), same data model.

In [ ]:
# LOCAL SIMULATION — Kafka-like stream processing with Python queues
# Illustrates the Lambda/Kappa concept without actual Kafka/Flink

import queue, time, threading
from collections import defaultdict
from datetime import datetime

# Simulate Kafka as a simple queue
kafka_topic = queue.Queue()

# Simulate incoming telemetry events
SAMPLE_EVENTS = [
    {"server_id": "srv-01", "cpu": 72.5, "ts": "2026-02-27T10:00:00"},
    {"server_id": "srv-02", "cpu": 91.3, "ts": "2026-02-27T10:00:01"},
    {"server_id": "srv-01", "cpu": 75.1, "ts": "2026-02-27T10:00:02"},
    {"server_id": "srv-03", "cpu": 45.2, "ts": "2026-02-27T10:00:03"},
    {"server_id": "srv-02", "cpu": 93.0, "ts": "2026-02-27T10:00:04"},
    {"server_id": "srv-01", "cpu": 80.4, "ts": "2026-02-27T10:00:05"},
    {"server_id": "srv-03", "cpu": 48.7, "ts": "2026-02-27T10:00:06"},
]

# SPEED LAYER: real-time alert processor (like Flink job)
alerts_fired = []
def speed_layer(event):
    if event["cpu"] > 90:
        alert = f"ALERT: {event['server_id']} CPU={event['cpu']}% at {event['ts']}"
        alerts_fired.append(alert)

# BATCH LAYER: aggregate at end (like Spark job over micro-batch)
batch_buffer = defaultdict(list)
s3_sink = []    # what would be written to S3 Parquet

def batch_layer(events):
    for e in events:
        batch_buffer[e["server_id"]].append(e["cpu"])
    for server_id, cpus in batch_buffer.items():
        s3_sink.append({
            "server_id": server_id,
            "avg_cpu": round(sum(cpus) / len(cpus), 2),
            "max_cpu": max(cpus),
            "sample_count": len(cpus)
        })

# ── Process the stream ───────────────────────────────────────────────────────
print("Processing telemetry stream...")
for event in SAMPLE_EVENTS:
    kafka_topic.put(event)

all_events = []
while not kafka_topic.empty():
    event = kafka_topic.get()
    all_events.append(event)
    speed_layer(event)   # real-time processing

batch_layer(all_events)  # batch processing

print()
print("SPEED LAYER output (real-time alerts):")
for alert in alerts_fired:
    print(f"  {alert}")

print()
print("BATCH LAYER output (would write to S3 Parquet):")
for row in sorted(s3_sink, key=lambda x: x["server_id"]):
    print(f"  {row}")

## Interview Q&A

**Q: Why would you choose Kappa over Lambda?**
A: Simpler maintenance — one codebase, one team, no drift between batch and stream logic. Modern stream processors (Flink, Spark Structured Streaming) handle complex aggregations, joins, and windowing well enough to replace batch. Choose Kappa when the stream processor can handle all your computation requirements.

**Q: What is "late data" and how do you handle it?**
A: Events that arrive after their processing window has closed. Solutions: watermarks (tolerate N minutes of lateness), upsert to the target table for late corrections, reprocess from Kafka for large late-data volumes.

**Q: What is exactly-once semantics?**
A: Guarantee that each event is processed exactly once — not duplicated, not lost. Hard to achieve in distributed systems. At-least-once + idempotent sinks is a practical alternative: process events at least once, but make the write operation idempotent (same result whether written once or multiple times).

**Q: How would you design a pipeline for 6,000 endpoints producing 1 metric per second?**
A: 6,000 events/sec → Kinesis or Kafka (10 shards/partitions) → Flink or Lambda (real-time alerts, 5-min windowed aggregates) → S3 micro-batches in Parquet → Glue crawler → Athena. Daily Glue job for historical analytics. CloudWatch for orchestration.

**Q: What is a watermark in stream processing?**
A: A timestamp marker that tells the stream processor "all events up to time T have arrived." Events beyond the watermark in lateness are dropped or processed as late. The watermark moves forward as event time advances.

## The Citi Angle

**This architecture is directly from Citi experience.** Managing 4 APM tools with different data formats was the core challenge.

**The unified telemetry pipeline (real architecture):**
```
4 APM Tools (CA APM, AppDynamics, Dynatrace, BMC TrueSight)
    │ REST API polling (30-min cadence) + agent push (5-min cadence)
    ▼
Custom Python collectors (one per APM tool)
    │ normalize to common schema: {server_id, metric, value, timestamp}
    ▼
Message queue (RabbitMQ in practice, equivalent to Kafka)
    │
    ├──→ Real-time: threshold checks → email + ITSM ticket
    └──→ Batch sink: PostgreSQL + file exports → analytics
                │
                ▼
    Python ETL (pandas) → CSV → Excel reports (manual era)
    Later: S3 → Athena SQL → automated dashboards (target state)
```

**What I'd do differently with modern AWS:**
- Replace RabbitMQ with Kinesis Data Streams
- Replace Python batch ETL with Glue jobs
- Replace PostgreSQL analytics tables with S3 Parquet + Athena
- Add Flink for real-time aggregation instead of threshold-only checks

**Interview line:** *"At Citi, we had a Lambda-style architecture without knowing the name — batch aggregation in nightly jobs and a real-time alerting path from the same event stream. The lesson: keep the two codebases in sync. In Kappa, you get this for free by replaying from the source."*